<a href="https://colab.research.google.com/github/LGLV-Ciencia-de-Datos/Curso_python_Ciencia_de_Datos/blob/main/d)_Model_Validation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Validación del Modelo** (Model Validation)
Mide el rendimiento de tu modelo para poder probar y comparar alternativas.

### Has construido un modelo, ¿pero qué tan bueno es?
En esta lección, aprenderás a usar la validación del modelo para medir la calidad de tu modelo. Medir la calidad del modelo es la clave para mejorar tus modelos de forma iterativa.

### Qué es la validación del modelo?

Querrás evaluar casi todos los modelos que construyas. En la mayoría de las aplicaciones (aunque no en todas), la medida relevante de la calidad del modelo es la precisión predictiva. En otras palabras, si las predicciones del modelo se acercarán a lo que realmente sucede.

Mucha gente comete un gran error al medir la precisión predictiva. Hacen predicciones con sus datos de entrenamiento y comparan esas predicciones con los valores objetivo en los datos de entrenamiento. Verás el problema con este enfoque y cómo resolverlo en un momento, pero primero, pensemos en cómo haríamos esto.

Primero, necesitarías resumir la calidad del modelo de una manera comprensible. Si comparas los valores de vivienda predichos y reales para 10,000 casas, es probable que encuentres una mezcla de predicciones buenas y malas. Mirar una lista de 10,000 valores predichos y reales no tendría sentido. Necesitamos resumir esto en una sola métrica.

Existen muchas métricas para resumir la calidad del modelo, pero comenzaremos con una llamada Error Absoluto Medio `Mean Absolute Error`(también conocido como `MAE`). Desglosemos esta métrica comenzando con la última palabra, "error".

### El error de predicción para cada casa es:


In [ ]:
error = actual_predicted

Entonces, si una casa costó `$150,000`
y predijiste que costaría `$100,000`, el error es de $50,000.

Con la métrica `MAE` (Error Absoluto Medio), tomamos el valor absoluto de cada error. Esto convierte cada error en un número positivo. Luego, sacamos el promedio de esos errores absolutos. Esta es nuestra medida de la calidad del modelo. En palabras sencillas, se puede decir como:

* En promedio, nuestras predicciones se desvían en X.

Para calcular el `MAE`, primero necesitamos un modelo. Este se construye en una celda oculta a continuación, que puedes revisar haciendo clic en el botón de código.



In [ ]:
import pandas as pd
import kagglehub
import numpy as np

# Download latest version
path = kagglehub.dataset_download("dansbecker/melbourne-housing-snapshot")

print("Path to dataset files:", path)

# Load the data into a pandas DataFrame
melbourne_file_path = path + '/melb_data.csv'
melbourne_data = pd.read_csv(melbourne_file_path)

# Print the first 5 rows of the DataFrame
# display(melbourne_data)

# Filter rows with missing price values
filtered_melbourne_data = melbourne_data.dropna(axis=0)
# Choose target and features
y = filtered_melbourne_data.Price
melbourne_features = ['Rooms', 'Bathroom', 'Landsize', 'BuildingArea',
                        'YearBuilt', 'Lattitude', 'Longtitude']
X = filtered_melbourne_data[melbourne_features]

from sklearn.tree import DecisionTreeRegressor
# Define model
melbourne_model = DecisionTreeRegressor()
# Fit model
melbourne_model.fit(X, y)

Una vez que tenemos un modelo, así es como se calcula el error absoluto medio `MAE`:

In [ ]:
from sklearn.metrics import mean_absolute_error

predicted_home_prices = melbourne_model.predict(X)
mean_absolute_error(y, predicted_home_prices)

### El problema con los puntajes "dentro de la muestra"

La medida que acabamos de calcular puede llamarse un puntaje "dentro de la muestra" `in-sample`. Usamos una única "muestra" de casas tanto para construir el modelo como para evaluarlo. Aquí te explicamos por qué esto es un problema.

Imagina que, en el gran mercado inmobiliario, el color de la puerta no tiene relación con el precio de la casa.

Sin embargo, en la muestra de datos que utilizaste para construir el modelo, todas las casas con puertas verdes eran muy caras. La tarea del modelo es encontrar patrones que predigan los precios de las casas, por lo que verá este patrón y siempre predecirá precios altos para las casas con puertas verdes.

Dado que este patrón se derivó de los datos de entrenamiento, el modelo parecerá preciso con los datos de entrenamiento.

Pero si este patrón no se mantiene cuando el modelo ve datos nuevos, el modelo sería muy inexacto cuando se use en la práctica.

Dado que el valor práctico de los modelos proviene de hacer predicciones sobre datos nuevos, medimos el rendimiento con datos que no se utilizaron para construir el modelo. La forma más sencilla de hacerlo es excluir algunos datos del proceso de construcción del modelo y luego usarlos para probar la precisión del modelo con datos que no ha visto antes. Estos datos se llaman datos de validación `validation data`.


### Codificándolo

La biblioteca `scikit-learn` tiene una función, `train_test_split`, para dividir los datos en dos partes. Usaremos una de esas partes como datos de entrenamiento para ajustar el modelo, y la otra parte como datos de validación para calcular el error absoluto medio (`mean_absolute_error`).

In [ ]:
from sklearn.model_selection import train_test_split

# Dividir los datos en conjuntos de entrenamiento y validación, tanto para las
# características (features X) como para el objetivo (target y).
# La división se basa en un generador de números aleatorios. Suministrar un valor
# numérico al argumento random_state garantiza que obtengamos la misma división
# cada vez que ejecutemos este script.

train_X, val_X, train_y, val_y = train_test_split(X, y, random_state = 0)
# Definir el modelo

melbourne_model = DecisionTreeRegressor()
# Entrenar el modelo
melbourne_model.fit(train_X, train_y)

# Obtener los precios predichos en los datos de validación
val_predictions = melbourne_model.predict(val_X)
print(mean_absolute_error(val_y, val_predictions))
MAE = mean_absolute_error(val_y, val_predictions)

¡Increíble!

El error absoluto medio para los datos "dentro de la muestra" `in-sample` fue de aproximadamente `500 dólares`. Mientras que, con datos de validación `validation data`, fuera de la muestra, es de más de `250,000 dólares`.

Esta es la diferencia entre un modelo que es casi exacto y uno que es inútil para la mayoría de los propósitos prácticos. Como punto de referencia, el valor promedio de las casas en los datos de validación es de 1.1 millones de dólares. Por lo tanto, el error en los nuevos datos es aproximadamente una cuarta parte del valor promedio de las casas.

Hay muchas maneras de mejorar este modelo, como experimentar para encontrar mejores características o diferentes tipos de modelos.


In [ ]:
# valor promedio de las casas en los datos de validación
val_y.mean()

In [ ]:
# Valor promedio de las casas.
mean_hoses_val = melbourne_data.Price.mean()
mean_hoses_val

In [ ]:
# el error en los nuevos datos es aproximadamente una cuarta parte del valor promedio de las casas.
error_nuevos_datos = MAE/mean_hoses_val
error_nuevos_datos